In [8]:
import pandas as pd
import numpy as np

listings = pd.read_csv("/content/listings.csv", sep=None, engine='python', on_bad_lines='skip')
rates = pd.read_csv("/content/past_rates.csv")

In [9]:
print(rates.columns)
rates_agg = rates.groupby("listing_id").agg({
    "revenue": "mean",
    "occupancy": "mean"
}).reset_index()

Index(['listing_id', 'date', 'vacant_days', 'reserved_days', 'occupancy',
       'revenue', 'rate_avg', 'booked_rate_avg', 'booking_lead_time_avg',
       'length_of_stay_avg', 'min_nights_avg', 'native_booked_rate_avg',
       'native_rate_avg', 'native_revenue', 'country', 'state', 'city'],
      dtype='object')


In [10]:
listings['listing_id'] = pd.to_numeric(listings['listing_id'], errors='coerce')
listings.dropna(subset=['listing_id'], inplace=True)
listings['listing_id'] = listings['listing_id'].astype(int)
df = pd.merge(
    listings,
    rates_agg,
    on="listing_id",
    how="inner"   # or 'left'
)

In [11]:
# ================================
# 4. HANDLE COLUMN NAMES (IMPORTANT)
# ================================
# Ensure consistent key column

if 'id' in listings.columns:
    listings.rename(columns={'id': 'listing_id'}, inplace=True)

if 'id' in rates.columns:
    rates.rename(columns={'id': 'listing_id'}, inplace=True)

In [12]:
# ================================
# 5. AGGREGATE TIME-SERIES DATA
# ================================
# Convert multiple rows per listing → single row

rates_agg = rates.groupby("listing_id").agg({
    "revenue": "mean",
    "occupancy": "mean"
}).reset_index()

rates_agg.rename(columns={
    "revenue": "avg_revenue",
    "occupancy": "avg_occupancy"
}, inplace=True)

print(rates_agg.head())

   listing_id  avg_revenue  avg_occupancy
0        2818   876.083333       0.317750
1        3176   511.083333       0.194833
2        5396  2114.416667       0.499000
3        6020  1147.916667       0.430583
4        6022  1420.666667       0.476667


In [13]:
# ================================
# 6. MERGE DATASETS
# ================================
df = pd.merge(
    listings,
    rates_agg,
    on="listing_id",
    how="left"
)

print("Merged Shape:", df.shape)

Merged Shape: (95329, 63)


In [14]:
# ================================
# 7. HANDLE MISSING VALUES
# ================================

# Numerical columns
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col].fillna("Unknown", inplace=True)

print("Missing values after cleaning:")
print(df.isnull().sum())

/tmp/ipykernel_6286/2712877038.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_6286/2712877038.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

Missing values after cleaning:
listing_id         0
listing_type       0
room_type          0
cover_photo_url    0
photos_count       0
                  ..
country            0
state              0
city               0
avg_revenue        0
avg_occupancy      0
Length: 63, dtype: int64


In [15]:
# ================================
# 8. CLEAN AMENITIES COLUMN
# ================================

if "amenities" in df.columns:
    # Convert text to count of amenities
    df["amenities_count"] = df["amenities"].apply(
        lambda x: len(str(x).split(",")) if x != "Unknown" else 0
    )

In [16]:
# ================================
# 9. OUTLIER REMOVAL (IQR METHOD)
# ================================

def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[col] >= lower) & (df[col] <= upper)]

# Apply on important columns
for col in ["price", "avg_price", "avg_occupancy"]:
    if col in df.columns:
        df = remove_outliers(df, col)

print("Shape after outlier removal:", df.shape)

Shape after outlier removal: (95290, 64)


In [17]:
# ================================
# 10. FEATURE ENGINEERING
# ================================

# Revenue estimate
if "avg_price" in df.columns and "avg_occupancy" in df.columns:
    df["estimated_revenue"] = df["avg_price"] * df["avg_occupancy"]

# Price per person
if "accommodates" in df.columns:
    df["price_per_person"] = df["price"] / df["accommodates"]

# Fill infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# ================================
# 12. SAVE CLEANED DATA
# ================================
import os

# Create the directory if it doesn't exist
output_dir = "content/data/processed"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "cleaned_airbnb_data.csv"), index=False)

print("✅ Cleaned dataset saved successfully!")

# We merged the listings and time-series datasets by first aggregating the time-series
# data to a listing level, then handled missing values using median and mode strategies,
# removed outliers using the IQR method, and created new features like estimated revenue
# and amenities count to enhance analysis.

In [25]:
df

,listing_id,listing_type,room_type,cover_photo_url,photos_count,host_id,superhost,latitude,longitude,guests,...,l90d_reserved_days,l90d_blocked_days,l90d_available_days,l90d_total_days,country,state,city,avg_revenue,avg_occupancy,amenities_count
0,121902,Entire home,entire_home,https://a0.muscache.com/im/pictures/77c0e3a9-0...,77.0,fe453949b595,False,37.0758,27.2426,6.0,...,0.0,90.0,90.0,90.0,Turkey,Muğla,Bodrum,2037.916667,0.161250,75
1,805342,Entire condo,entire_home,https://a0.muscache.com/im/pictures/11494599/4...,16.0,59711ec4c245,False,37.0092,27.2563,3.0,...,0.0,0.0,90.0,90.0,Turkey,Muğla,Bodrum,697.666667,0.250000,33
2,805361,Entire home,entire_home,https://a0.muscache.com/im/pictures/bda48dbc-d...,34.0,d217bf6e3427,False,37.0292,27.4410,5.0,...,0.0,0.0,90.0,90.0,Turkey,Muğla,Bodrum,1883.333333,0.172833,44
3,853827,Entire villa,entire_home,https://a0.muscache.com/im/pictures/26626113/a...,70.0,605fc7d80e02,False,37.0434,27.2517,16.0,...,2.0,0.0,88.0,90.0,Turkey,Muğla,Bodrum,6769.833333,0.318000,18
4,967193,Entire villa,entire_home,https://a0.muscache.com/im/pictures/68107669/2...,37.0,3b963e8cd040,False,37.0429,27.3898,4.0,...,0.0,0.0,90.0,90.0,Turkey,Muğla,Bodrum,384.833333,0.054333,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95324,46309704,Entire rental unit,entire_home,https://a0.muscache.com/im/pictures/hosting/Ho...,33.0,6f93ff585c4d,False,46.5159,6.6059,6.0,...,0.0,0.0,90.0,90.0,Switzerland,Vaud,Lausanne,2261.416667,0.231500,33
95325,46488808,Entire rental unit,entire_home,https://a0.muscache.com/im/pictures/hosting/Ho...,10.0,dfbbfa03eaf3,False,46.5170,6.6442,2.0,...,12.0,66.0,78.0,90.0,Switzerland,Vaud,Lausanne,1698.833333,0.347583,23
95326,46595801,Entire rental unit,entire_home,https://a0.muscache.com/im/pictures/prohost-ap...,13.0,09e353c919ae,False,46.5214,6.6191,5.0,...,0.0,90.0,90.0,90.0,Switzerland,Vaud,Lausanne,1214.833333,0.164833,33
95327,46637115,Entire rental unit,entire_home,https://a0.muscache.com/im/pictures/prohost-ap...,15.0,09e353c919ae,False,46.5167,6.6075,6.0,...,8.0,0.0,82.0,90.0,Switzerland,Vaud,Lausanne,2551.333333,0.275083,33


df


In [29]:
important_cols = [
    "listing_id",
    "city",
    "state",
    "country",

    "listing_type",
    "room_type",

    "bedrooms",
    "guests",

    "avg_revenue",
    "avg_occupancy",

    "amenities_count",

    "rating_overall",   # if exists
    "rating_location",  # if exists


]

In [30]:
df = df[important_cols].copy()

In [31]:
df

,listing_id,city,state,country,listing_type,room_type,bedrooms,guests,avg_revenue,avg_occupancy,amenities_count,rating_overall,rating_location
0,121902,Bodrum,Muğla,Turkey,Entire home,entire_home,3.0,6.0,2037.916667,0.161250,75,5.00,4.9
1,805342,Bodrum,Muğla,Turkey,Entire condo,entire_home,1.0,3.0,697.666667,0.250000,33,5.00,5.0
2,805361,Bodrum,Muğla,Turkey,Entire home,entire_home,2.0,5.0,1883.333333,0.172833,44,4.76,4.9
3,853827,Bodrum,Muğla,Turkey,Entire villa,entire_home,12.0,16.0,6769.833333,0.318000,18,4.85,4.6
4,967193,Bodrum,Muğla,Turkey,Entire villa,entire_home,2.0,4.0,384.833333,0.054333,12,4.82,4.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95324,46309704,Lausanne,Vaud,Switzerland,Entire rental unit,entire_home,2.0,6.0,2261.416667,0.231500,33,4.70,4.6
95325,46488808,Lausanne,Vaud,Switzerland,Entire rental unit,entire_home,1.0,2.0,1698.833333,0.347583,23,4.75,5.0
95326,46595801,Lausanne,Vaud,Switzerland,Entire rental unit,entire_home,3.0,5.0,1214.833333,0.164833,33,4.71,4.6
95327,46637115,Lausanne,Vaud,Switzerland,Entire rental unit,entire_home,3.0,6.0,2551.333333,0.275083,33,4.61,4.8


In [33]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
df.fillna(df.median(numeric_only=True), inplace=True)
df.fillna("Unknown", inplace=True)

/tmp/ipykernel_6286/2849145427.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)
/tmp/ipykernel_6286/2849145427.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.fillna(df.median(numeric_only=True), inplace=True)
/tmp/ipykernel_6286/2849145427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.fillna("Unknown", inplace=True)


In [34]:
df["ADR"] = df["avg_revenue"] / (df["avg_occupancy"] + 1e-5)

/tmp/ipykernel_6286/2587064907.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["ADR"] = df["avg_revenue"] / (df["avg_occupancy"] + 1e-5)


In [36]:
df["RevPAR"] = df["ADR"] * df["avg_occupancy"]

/tmp/ipykernel_6286/505188124.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["RevPAR"] = df["ADR"] * df["avg_occupancy"]


In [37]:
df["revenue_category"] = pd.qcut(
    df["avg_revenue"],
    q=3,
    labels=["Low", "Medium", "High"]
)

/tmp/ipykernel_6286/2033989385.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["revenue_category"] = pd.qcut(


In [38]:
df.head()

,listing_id,city,state,country,listing_type,room_type,bedrooms,guests,avg_revenue,avg_occupancy,amenities_count,rating_overall,rating_location,ADR,RevPAR,revenue_category
0,121902,Bodrum,Muğla,Turkey,Entire home,entire_home,3.0,6.0,2037.916667,0.161250,75,5.00,4.9,12637.459176,2037.790292,High
1,805342,Bodrum,Muğla,Turkey,Entire condo,entire_home,1.0,3.0,697.666667,0.250000,33,5.00,5.0,2790.555044,697.638761,Medium
2,805361,Bodrum,Muğla,Turkey,Entire home,entire_home,2.0,5.0,1883.333333,0.172833,44,4.76,4.9,10896.187299,1883.224371,High
3,853827,Bodrum,Muğla,Turkey,Entire villa,entire_home,12.0,16.0,6769.833333,0.318000,18,4.85,4.6,21288.114630,6769.620452,High
4,967193,Bodrum,Muğla,Turkey,Entire villa,entire_home,2.0,4.0,384.833333,0.054333,12,4.82,4.8,7081.518739,384.762518,Low
